<a href="https://colab.research.google.com/github/Dilandds/CNN-emotion-detection/blob/main/CCN_emotion_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Facial Emotion Recognition — CNN (PyTorch)

Classifying facial expressions into 7 emotions using the FER2013 dataset,
with a convolutional neural network built from scratch in PyTorch.

**Plan:** environment & data → explore → data pipeline → model → training loop →
train → evaluate → write-up.

## Step 1 — Environment + Data

Before running: **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save.**

### 1.1 Confirm the GPU is connected

`torch.cuda.is_available()` asks whether PyTorch can see a GPU. We define `device`
once here and send the model and every batch to it later, so the rest of the
notebook runs unchanged on CPU or GPU.

You should see `CUDA available: True` before going further — on CPU this trains
in hours instead of minutes.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

### 1.2 Kaggle credentials

Get the key: kaggle.com → profile → Settings → API → **Create New Token**.
That downloads `kaggle.json`. Run the cell below and upload that file.

The Kaggle CLI looks for the key at `~/.kaggle/kaggle.json`. `chmod 600` makes it
readable only by you — Kaggle refuses to run if the permissions are looser, which
is the most common first-time error here.

In [ ]:
from google.colab import files

files.upload()  # select the kaggle.json you just downloaded

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

### 1.3 Download and unzip FER2013

This is the image-folder version of FER2013: already split into `train/` and
`test/`, one subfolder per emotion. Much easier to work with than the original
CSV of pixel strings.

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d msambare/fer2013
!unzip -q fer2013.zip -d /content/fer2013

### 1.4 Verify folder structure and image counts per class

Walks each split, lists the class subfolders, and counts the files in each.

`sorted()` matters: PyTorch's `ImageFolder` assigns label indices in sorted
alphabetical order, so this ordering **is** our label mapping —
angry=0, disgust=1, fear=2, happy=3, neutral=4, sad=5, surprise=6.

Expected: 7 classes, ~28,709 train images and ~7,178 test images.

In [ ]:
import os

DATA_DIR = "/content/fer2013"

for split in ["train", "test"]:
    split_dir = os.path.join(DATA_DIR, split)
    classes = sorted(os.listdir(split_dir))
    print(f"\n{split.upper()}  ({len(classes)} classes)")
    total = 0
    for c in classes:
        n = len(os.listdir(os.path.join(split_dir, c)))
        total += n
        print(f"  {c:10s} {n:6d}")
    print(f"  {'TOTAL':10s} {total:6d}")